In [36]:
# !pip install gspread pandas google-auth

In [37]:
import os

os.environ["GOOGLE_SHEETS_CREDENTIALS"] = (
    "/opt/notebooks/psalms_nlp_sp26/private/psalms-blind-scoring-f39a2660c9de.json"
)

os.getcwd()

'/opt/notebooks/psalms_nlp_sp26/query_compare'

In [38]:
from pathlib import Path

# Current notebook directory
notebook_dir = Path.cwd()

# Build path to the JSON
cred_path = notebook_dir.parent / "private" / "psalms-blind-scoring-f39a2660c9de.json"

print("Credential path exists?", cred_path.exists())



Credential path exists? True


In [39]:
os.getcwd()

'/opt/notebooks/psalms_nlp_sp26/query_compare'

In [40]:
import gspread
from google.oauth2.service_account import Credentials

creds = Credentials.from_service_account_file(
    os.environ["GOOGLE_SHEETS_CREDENTIALS"],
    scopes=[
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive"
    ]
)

client = gspread.authorize(creds)

sheet = client.open("results_scored")

In [41]:
import pandas as pd

# Access second sheet (index 1)
worksheet2 = sheet.get_worksheet(1)

# Get all values
data = worksheet2.get_all_values()

# Convert to DataFrame (first row as header)
df = pd.DataFrame(data[1:], columns=data[0])
df


,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,CadenScore,Score1,User1,Score2,User2,Score3,User3
0,For the Peace of the world,TFIDF_GLoVe,43.73,Bible,92,1For the day before the Sabbath when the earth...,4,,,,,,
1,For the Peace of the world,TFIDF_GLoVe,41,Psalter,121,"I was glad because of them that said to me, Le...",0,,,,,,
2,For the Peace of the world,TFIDF_GLoVe,40.99,Bible,121,1An ode of ascents Iwas glad when they said to...,8,,,,,,
3,For the Peace of the world,TFIDF_GLoVe,38.74,Bible,131,1An ode of ascents Remember David O Lord And a...,4,,,,,,
4,For the Peace of the world,TFIDF_GLoVe,38.43,Bible,96,By David when His earth is restored The Lord r...,9,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,"Have mercy on me, O God, have mercy on me. For...",TFIDF,21.9,Psalter,39,"I waited patiently for the Lord, and He inclin...",9,,,,,,
188,"Have mercy on me, O God, have mercy on me. For...",TFIDF,21.33,Psalter,54,"Give ear to my prayer, O God, and despise not ...",6,,,,,,
189,"Have mercy on me, O God, have mercy on me. For...",TFIDF,20.87,Psalter,142,"Hear my prayer, O Lord; give ear unto my suppl...",9,8,p01,,,,
190,"Have mercy on me, O God, have mercy on me. For...",TFIDF,19.55,Psalter,68,"Save me, O God, for the waters are come in unt...",4,,,,,,


# Preparing the data
I want each row to hold one of the four possible scored for each of the `192 results`

I need to start by unpivoting my own score separte from the other scores.

In [42]:
caden = df[df.columns[:7]]

In [43]:
caden['User'] = 'caden'

caden = caden.rename(columns={"CadenScore": "Score"})

caden

/tmp/ipykernel_268/3532660364.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  caden['User'] = 'caden'


,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,Score,User
0,For the Peace of the world,TFIDF_GLoVe,43.73,Bible,92,1For the day before the Sabbath when the earth...,4,caden
1,For the Peace of the world,TFIDF_GLoVe,41,Psalter,121,"I was glad because of them that said to me, Le...",0,caden
2,For the Peace of the world,TFIDF_GLoVe,40.99,Bible,121,1An ode of ascents Iwas glad when they said to...,8,caden
3,For the Peace of the world,TFIDF_GLoVe,38.74,Bible,131,1An ode of ascents Remember David O Lord And a...,4,caden
4,For the Peace of the world,TFIDF_GLoVe,38.43,Bible,96,By David when His earth is restored The Lord r...,9,caden
...,...,...,...,...,...,...,...,...
187,"Have mercy on me, O God, have mercy on me. For...",TFIDF,21.9,Psalter,39,"I waited patiently for the Lord, and He inclin...",9,caden
188,"Have mercy on me, O God, have mercy on me. For...",TFIDF,21.33,Psalter,54,"Give ear to my prayer, O God, and despise not ...",6,caden
189,"Have mercy on me, O God, have mercy on me. For...",TFIDF,20.87,Psalter,142,"Hear my prayer, O Lord; give ear unto my suppl...",9,caden
190,"Have mercy on me, O God, have mercy on me. For...",TFIDF,19.55,Psalter,68,"Save me, O God, for the waters are come in unt...",4,caden


In [44]:
caden  = caden[['Query', 'Method', 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'User', 'Score']]
caden

,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,For the Peace of the world,TFIDF_GLoVe,43.73,Bible,92,1For the day before the Sabbath when the earth...,caden,4
1,For the Peace of the world,TFIDF_GLoVe,41,Psalter,121,"I was glad because of them that said to me, Le...",caden,0
2,For the Peace of the world,TFIDF_GLoVe,40.99,Bible,121,1An ode of ascents Iwas glad when they said to...,caden,8
3,For the Peace of the world,TFIDF_GLoVe,38.74,Bible,131,1An ode of ascents Remember David O Lord And a...,caden,4
4,For the Peace of the world,TFIDF_GLoVe,38.43,Bible,96,By David when His earth is restored The Lord r...,caden,9
...,...,...,...,...,...,...,...,...
187,"Have mercy on me, O God, have mercy on me. For...",TFIDF,21.9,Psalter,39,"I waited patiently for the Lord, and He inclin...",caden,9
188,"Have mercy on me, O God, have mercy on me. For...",TFIDF,21.33,Psalter,54,"Give ear to my prayer, O God, and despise not ...",caden,6
189,"Have mercy on me, O God, have mercy on me. For...",TFIDF,20.87,Psalter,142,"Hear my prayer, O Lord; give ear unto my suppl...",caden,9
190,"Have mercy on me, O God, have mercy on me. For...",TFIDF,19.55,Psalter,68,"Save me, O God, for the waters are come in unt...",caden,4


Moving on to prepaering the external scores.

In [45]:
external = df[['Query', 'Method', 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'User1', 'Score1', 'User2', 'Score2', 'User3', 'Score3']]

external

,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,User1,Score1,User2,Score2,User3,Score3
0,For the Peace of the world,TFIDF_GLoVe,43.73,Bible,92,1For the day before the Sabbath when the earth...,,,,,,
1,For the Peace of the world,TFIDF_GLoVe,41,Psalter,121,"I was glad because of them that said to me, Le...",,,,,,
2,For the Peace of the world,TFIDF_GLoVe,40.99,Bible,121,1An ode of ascents Iwas glad when they said to...,,,,,,
3,For the Peace of the world,TFIDF_GLoVe,38.74,Bible,131,1An ode of ascents Remember David O Lord And a...,,,,,,
4,For the Peace of the world,TFIDF_GLoVe,38.43,Bible,96,By David when His earth is restored The Lord r...,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...
187,"Have mercy on me, O God, have mercy on me. For...",TFIDF,21.9,Psalter,39,"I waited patiently for the Lord, and He inclin...",,,,,,
188,"Have mercy on me, O God, have mercy on me. For...",TFIDF,21.33,Psalter,54,"Give ear to my prayer, O God, and despise not ...",,,,,,
189,"Have mercy on me, O God, have mercy on me. For...",TFIDF,20.87,Psalter,142,"Hear my prayer, O Lord; give ear unto my suppl...",p01,8,,,,
190,"Have mercy on me, O God, have mercy on me. For...",TFIDF,19.55,Psalter,68,"Save me, O God, for the waters are come in unt...",,,,,,


In [46]:
# Unpivot User/Score pairs
df_long = pd.wide_to_long(
    external,
    stubnames=["User", "Score"],  # the base column names
    i=["Query", "Method", "Similarity Score (%)", "Text", "Psalm Num", "Verse"],  # columns to keep
    j="Pair",  # new column for the pair number
    sep=""      # number comes directly after the stub name
).reset_index()

# Optional: reorder columns
df_long = df_long[["Query", "Method", "Similarity Score (%)", "Text", "Psalm Num", "Verse", "Pair", "User", "Score"]]



In [47]:
external = df_long[["Query", "Method", "Similarity Score (%)", "Text", "Psalm Num", "Verse", "User", "Score"]]

external.head()

,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,For the Peace of the world,TFIDF_GLoVe,43.73,Bible,92,1For the day before the Sabbath when the earth...,,
1,For the Peace of the world,TFIDF_GLoVe,43.73,Bible,92,1For the day before the Sabbath when the earth...,,
2,For the Peace of the world,TFIDF_GLoVe,43.73,Bible,92,1For the day before the Sabbath when the earth...,,
3,For the Peace of the world,TFIDF_GLoVe,41,Psalter,121,"I was glad because of them that said to me, Le...",,
4,For the Peace of the world,TFIDF_GLoVe,41,Psalter,121,"I was glad because of them that said to me, Le...",,


In [48]:
caden.head()

,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,For the Peace of the world,TFIDF_GLoVe,43.73,Bible,92,1For the day before the Sabbath when the earth...,caden,4
1,For the Peace of the world,TFIDF_GLoVe,41,Psalter,121,"I was glad because of them that said to me, Le...",caden,0
2,For the Peace of the world,TFIDF_GLoVe,40.99,Bible,121,1An ode of ascents Iwas glad when they said to...,caden,8
3,For the Peace of the world,TFIDF_GLoVe,38.74,Bible,131,1An ode of ascents Remember David O Lord And a...,caden,4
4,For the Peace of the world,TFIDF_GLoVe,38.43,Bible,96,By David when His earth is restored The Lord r...,caden,9


# Combining the Prepared Data back together

In [49]:
pd.concat([caden, external], ignore_index=True)

,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,For the Peace of the world,TFIDF_GLoVe,43.73,Bible,92,1For the day before the Sabbath when the earth...,caden,4
1,For the Peace of the world,TFIDF_GLoVe,41,Psalter,121,"I was glad because of them that said to me, Le...",caden,0
2,For the Peace of the world,TFIDF_GLoVe,40.99,Bible,121,1An ode of ascents Iwas glad when they said to...,caden,8
3,For the Peace of the world,TFIDF_GLoVe,38.74,Bible,131,1An ode of ascents Remember David O Lord And a...,caden,4
4,For the Peace of the world,TFIDF_GLoVe,38.43,Bible,96,By David when His earth is restored The Lord r...,caden,9
...,...,...,...,...,...,...,...,...
763,"Have mercy on me, O God, have mercy on me. For...",TFIDF,19.55,Psalter,68,"Save me, O God, for the waters are come in unt...",,
764,"Have mercy on me, O God, have mercy on me. For...",TFIDF,19.55,Psalter,68,"Save me, O God, for the waters are come in unt...",,
765,"Have mercy on me, O God, have mercy on me. For...",TFIDF,19.11,Psalter,70,"In Thee, O Lord, have I put my hope; let me ne...",,
766,"Have mercy on me, O God, have mercy on me. For...",TFIDF,19.11,Psalter,70,"In Thee, O Lord, have I put my hope; let me ne...",,
